# Activity 4: Function Calling, From the Inside

**Week 6 Day 3 · What actually happens when a model "calls a function"**

The model cannot query your database, look up a claim, or do arithmetic it does not trust itself with. It has no hands. What it *can* do is recognize when a question needs one of those things, stop, and tell you exactly what it wants run and with what arguments. You run it. You hand the result back. It continues.

That is the entire trick behind "function calling," "tool use," and, eventually, "agents." No magic, no code executing inside the model. In this notebook you will do every one of those steps by hand, one at a time, before you let a loop do it for you.

## What you will learn

- Why a model stops mid-task instead of answering (`finish_reason == "tool_calls"`)
- How to read what it is asking for and run it yourself
- How to hand the result back so the model can pick up where it left off
- Why a single request/response pair is not enough, and what a loop buys you

---
## Setup

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

---
# 1. Two things a model can't do on its own

A tiny claims database and a tiny policy database, both fictional, both something no model has memorized.

In [ ]:
CLAIMS_DB = {
    "CLM_101": {"status": "Approved", "amount": 3400.0, "type": "Auto Collision"},
    "CLM_102": {"status": "Approved", "amount": 1250.0, "type": "Property Loss"},
    "CLM_103": {"status": "Denied", "amount": 0.0, "type": "Fraud Flag"},
}

POLICIES_DB = {
    "POL_991": {"deductible": 500.0, "coverage": "Full Comprehensive"},
    "POL_992": {"deductible": 1000.0, "coverage": "Liability Only"},
}

CLAIM_TO_POLICY = {"CLM_101": "POL_991", "CLM_102": "POL_992"}

In [ ]:
def get_claim_status(claim_id: str) -> dict:
    """Look up an insurance claim's status, amount, and type."""
    return CLAIMS_DB.get(claim_id, {"error": f"{claim_id} not found"})


def get_policy_deductible(policy_id: str) -> dict:
    """Look up a policy's deductible amount and coverage type."""
    return POLICIES_DB.get(policy_id, {"error": f"{policy_id} not found"})


def calculate_net_payout(claim_amount: float, deductible: float) -> dict:
    """Subtract a deductible from a claim amount to get the net payout."""
    return {"net_payout": round(max(0.0, claim_amount - deductible), 2)}

Nothing about these three functions is special. They are plain Python, no decorators, no framework. The model has never seen `CLAIMS_DB`, will never see it, and cannot call `get_claim_status` directly, it can only *ask* for it. Making that ask legible to the model is the job of the schema in the next cell.

---
# 2. Describing a function without letting it run one

`tools` is a JSON description of each function: its name, what it does in plain English, and what arguments it needs. The model reads this like a menu. It does not receive the function's code, only its shape.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_claim_status",
            "description": "Look up an insurance claim's status, amount, and type by claim ID.",
            "parameters": {
                "type": "object",
                "properties": {"claim_id": {"type": "string", "description": "e.g. CLM_101"}},
                "required": ["claim_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_policy_deductible",
            "description": "Look up a policy's deductible amount by policy ID.",
            "parameters": {
                "type": "object",
                "properties": {"policy_id": {"type": "string", "description": "e.g. POL_991"}},
                "required": ["policy_id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_net_payout",
            "description": "Subtract a deductible from a claim amount to get the net payout.",
            "parameters": {
                "type": "object",
                "properties": {
                    "claim_amount": {"type": "number"},
                    "deductible": {"type": "number"},
                },
                "required": ["claim_amount", "deductible"],
            },
        },
    },
]

`AVAILABLE_FUNCTIONS` is the other half: a lookup from the *name* the model will send back to the *actual Python function* to run. The schema above is what the model reads. This dictionary is what you read.

In [ ]:
AVAILABLE_FUNCTIONS = {
    "get_claim_status": get_claim_status,
    "get_policy_deductible": get_policy_deductible,
    "calculate_net_payout": calculate_net_payout,
}

---
# 3. Watch it stop

Ask a question that needs `get_claim_status`, pass `tools`, and look at what comes back before printing anything friendly.

In [ ]:
messages = [{"role": "user", "content": "What is the status of claim CLM_101?"}]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    tools=tools,
)

reply = response.choices[0].message
print("finish_reason:", response.choices[0].finish_reason)
print("content:", reply.content)
print("tool_calls:", reply.tool_calls)

Read that output carefully before moving on.

`content` is `None`. The model did not answer the question. `finish_reason` is `"tool_calls"`, the value you were told to remember in Activity 1: the model is telling you, explicitly, "I stopped on purpose, I need something run first." And `tool_calls` contains exactly one entry: a function name (`get_claim_status`) and a JSON string of arguments (`{"claim_id": "CLM_101"}"`). The model figured out which function answers this question and what argument it needs, entirely from the `description` fields you wrote. It did not run anything. It handed you a work order.

---
# 4. Run the work order, by hand

No loop yet. Pull the one tool call out, run the matching Python function yourself, and look at the result.

In [ ]:
call = reply.tool_calls[0]
function_name = call.function.name
function_args = json.loads(call.function.arguments)

print("model wants:", function_name, function_args)

result = AVAILABLE_FUNCTIONS[function_name](**function_args)
print("result:", result)

That is the whole "execute" step. `AVAILABLE_FUNCTIONS[function_name]` looks up the real Python function by the name the model sent back, `**function_args` unpacks the arguments it asked for. You just did, by hand, everything a "tool executor" library does for you.

Now hand the result back. The model needs two new messages appended to the conversation: the assistant's own tool-call message (exactly as it came back, so it remembers what it asked for), and a `role="tool"` message carrying your result, tagged with the same `tool_call_id` so the model knows which request this answers.

In [ ]:
messages.append(reply)  # the assistant's tool-call request, unchanged
messages.append({
    "role": "tool",
    "tool_call_id": call.id,
    "content": json.dumps(result),
})

second_response = client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
final_reply = second_response.choices[0].message

print("finish_reason:", second_response.choices[0].finish_reason)
print(final_reply.content)

`finish_reason` is `"stop"` this time. The model had everything it needed and answered in plain language. You just completed one full function-calling round trip, by hand: **ask -> stop -> you run it -> hand it back -> answer.**

---
# 5. The problem with doing it by hand

That worked because the question needed exactly one tool, exactly once. Try this question instead, which needs the claim amount *and* the policy deductible *and* a calculation, three separate tool calls, two of which depend on results you do not have yet:

> "Claim CLM_101 is on policy POL_991. What is the net payout after the deductible?"

You could keep copy-pasting the four lines from Section 4 for every tool call, but you would not know in advance how many times, or in what order, until the model tells you. That is exactly the kind of "repeat until done" problem a loop exists for.

---
# 6. Generalize it: one function, any number of rounds

`run_conversation` does exactly what Sections 3 and 4 did by hand, except it keeps going: call the model, check `finish_reason`, if it says `tool_calls`, run every requested tool and append the results, then call again. It stops only when the model finally answers in plain text.

In [ ]:
def run_conversation(prompt, max_rounds=5):
    messages = [{"role": "user", "content": prompt}]

    for round_num in range(1, max_rounds + 1):
        response = client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=tools)
        reply = response.choices[0].message

        if response.choices[0].finish_reason != "tool_calls":
            print(f"[round {round_num}] model answered, stopping")
            return reply.content

        messages.append(reply)
        for call in reply.tool_calls:
            name = call.function.name
            args = json.loads(call.function.arguments)
            result = AVAILABLE_FUNCTIONS[name](**args)
            print(f"[round {round_num}] ran {name}({args}) -> {result}")
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

    return "Gave up after max_rounds without a final answer."

Two details worth pointing at. `for call in reply.tool_calls` handles the model asking for **more than one tool in the same round** (it can ask for `get_claim_status` and `get_policy_deductible` together, since neither depends on the other). And nothing stops the model from requesting the *same* function twice in different rounds with different arguments, `AVAILABLE_FUNCTIONS[name]` does not care, it just runs whatever it is told, whenever it is told.

Run it on the three-step question from Section 5.

In [ ]:
answer = run_conversation("Claim CLM_101 is on policy POL_991. What is the net payout after the deductible?")
print("\nFinal answer:", answer)

Watch the printed rounds. The model does not know the deductible until it has asked for it, and it cannot calculate the net payout until it has both numbers, so it genuinely cannot do this in one shot. The loop is what makes multi-step tool use possible at all; without it you would be back to copy-pasting Section 4 by hand for every round, and you would not know how many rounds to write in advance.

---
# Your Turn

Work in your own copy under `student-work/week6/day3/`.

1. Ask `run_conversation` a question that needs the **same tool called twice**, for example: *"What are the net payouts for claim CLM_101 on policy POL_991 and claim CLM_102 on policy POL_992?"* Watch the printed rounds and count how many times each function actually ran.
2. Add a fourth tool, `list_denied_claims()`, that takes no arguments and returns every claim in `CLAIMS_DB` with `status == "Denied"`. Add its schema to `tools` and its entry to `AVAILABLE_FUNCTIONS`, then ask `run_conversation("Which claims were denied and why might that be?")`.

**Stretch goal:** what happens if you ask a question that needs no tools at all, like `"What is 'liability' in plain English?"`? Trace through `run_conversation` and confirm it still works without ever entering the `tool_calls` branch.

## What you did

- Watched a model stop mid-task and hand back a structured request instead of an answer.
- Executed a requested function by hand and returned the result with the correct `tool_call_id`.
- Generalized the by-hand steps into a loop that keeps going until the model is satisfied.
- Ran a question that genuinely required multiple, dependent tool calls.

**Next:** [Activity 5](./Activity_5_Prompt_Engineering_and_ReAct.ipynb) starts from prompt design fundamentals, then builds a ReAct agent, a model that narrates its own reasoning between tool calls instead of calling them silently like `run_conversation` did here.